# 2 · Portfolio Construction

Builds two portfolios:
1. **Equal-weight** — naive 1/N benchmark  
2. **Max-Sharpe** — Markowitz optimisation using `scipy.optimize.minimize` (SLSQP)

**Key fix vs previous code:**  
The risk-free rate used in the Sharpe objective is `RF_MONTHLY` (6.5% ÷ 12),  
not a daily rate divided by 252 applied to monthly returns.

In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from pathlib import Path
import sys
sys.path.insert(0, '..')
from utils import (
    ANALYSIS_DIR, RF_MONTHLY, PERIODS_PER_YEAR,
    portfolio_return, portfolio_volatility,
    sharpe_ratio, neg_sharpe,
    annualise_return, annualise_vol
)

analysis_path = Path(ANALYSIS_DIR)

returns = pd.read_csv(
    analysis_path / "india_equity_returns.csv",
    index_col="Date", parse_dates=True
)

mean_returns = returns.mean().values
cov_matrix   = returns.cov().values
tickers      = returns.columns.tolist()
n            = len(tickers)

print(f"Assets          : {n}")
print(f"RF (monthly)    : {RF_MONTHLY*100:.4f}%")

# ── 1. Equal-weight portfolio ────────────────────────────────────────────────
w_equal = np.full(n, 1.0 / n)

# ── 2. Maximum-Sharpe optimisation ───────────────────────────────────────────
constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]
bounds      = tuple((0.0, 1.0) for _ in range(n))   # long-only
init_w      = np.full(n, 1.0 / n)

# Run from multiple starting points to avoid local optima
best_sharpe = -np.inf
w_opt       = init_w.copy()

np.random.seed(42)
for _ in range(50):
    x0  = np.random.dirichlet(np.ones(n))
    res = minimize(
        neg_sharpe, x0,
        args=(mean_returns, cov_matrix, RF_MONTHLY),
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"ftol": 1e-12, "maxiter": 1000}
    )
    s = -res.fun
    if res.success and s > best_sharpe:
        best_sharpe = s
        w_opt       = res.x.copy()

# Clip tiny negative weights from numerical noise
w_opt = np.clip(w_opt, 0, 1)
w_opt /= w_opt.sum()

# ── 3. Minimum-variance portfolio ────────────────────────────────────────────
def port_var(w):
    return portfolio_volatility(w, cov_matrix) ** 2

res_minvar = minimize(
    port_var, init_w,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
    options={"ftol": 1e-12, "maxiter": 1000}
)
w_minvar = np.clip(res_minvar.x, 0, 1)
w_minvar /= w_minvar.sum()

# ── 4. Summarise ─────────────────────────────────────────────────────────────
def describe(label, w):
    r  = portfolio_return(w, mean_returns)
    v  = portfolio_volatility(w, cov_matrix)
    sr = sharpe_ratio(w, mean_returns, cov_matrix, RF_MONTHLY)
    return {
        "Portfolio"        : label,
        "Monthly_Return"   : round(r, 6),
        "Monthly_Vol"      : round(v, 6),
        "Monthly_Sharpe"   : round(sr, 6),
        "Annual_Return"    : round(annualise_return(r), 6),
        "Annual_Vol"       : round(annualise_vol(v), 6),
        "Annual_Sharpe"    : round(sr * np.sqrt(PERIODS_PER_YEAR), 6),
    }

summary = pd.DataFrame([
    describe("Equal Weight",  w_equal),
    describe("Max Sharpe",    w_opt),
    describe("Min Variance",  w_minvar),
])

# ── 5. Save ───────────────────────────────────────────────────────────────────
weights_df = pd.DataFrame({
    "Asset"            : tickers,
    "Equal_Weight"     : w_equal,
    "MaxSharpe_Weight" : w_opt,
    "MinVar_Weight"    : w_minvar,
})

weights_df.to_csv(analysis_path / "portfolio_weights.csv", index=False)
summary.to_csv(analysis_path / "portfolio_summary.csv", index=False)

print("\nPortfolio Summary:")
print(summary.to_string(index=False))
print("\nWeights:")
print(weights_df.to_string(index=False))
print("\nSaved to analysis/")

Assets          : 9
RF (monthly)    : 0.5417%

Portfolio Summary:
   Portfolio  Monthly_Return  Monthly_Vol  Monthly_Sharpe  Annual_Return  Annual_Vol  Annual_Sharpe
Equal Weight        0.016574     0.048740        0.228923       0.218063    0.168840       0.793012
  Max Sharpe        0.022499     0.051766        0.329995       0.306039    0.179324       1.143135
Min Variance        0.012335     0.040125        0.172426       0.158490    0.138997       0.597301

Weights:
Asset  Equal_Weight  MaxSharpe_Weight  MinVar_Weight
 BRTI      0.111111          0.559715   1.980211e-01
 EICH      0.111111          0.089974   1.031682e-02
 HDBK      0.111111          0.000000   1.890161e-01
  HLL      0.111111          0.000000   3.780338e-01
 LART      0.111111          0.000000   2.735643e-02
 RELI      0.111111          0.000000   6.229006e-02
  SUN      0.111111          0.134901   1.349657e-01
 TISC      0.111111          0.052097   4.028490e-18
 TITN      0.111111          0.163312   1.57510